# Fase 3 — janela curta e recortes antigos

O mesmo serviço Python do menu 9 conserva seis turnos recentes. Quando um turno sai da janela, grava um **resumo extrativo**: a fala do usuário, sem a resposta do modelo. A busca abaixo usa palavras em comum, não embeddings nem FAISS. Os textos são fixtures locais; este notebook não prova compreensão por LLM.


In [ ]:
from pathlib import Path
import sys, tempfile
raiz = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(raiz / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.memoria import MemoriaConversas


## 1. Um dado sai da janela
As respostas abaixo são apenas placeholders de teste. O recorte conserva a frase original do usuário e seu ID de turno, sem inferir que Alfa seja um cadastro válido.


In [ ]:
pasta = tempfile.TemporaryDirectory()
caminho = Path(pasta.name) / 'resumos.sqlite3'
ana, bruno = Contexto('demo', 'ana'), Contexto('demo', 'bruno')
with Banco(caminho) as banco:
    banco.preparar_demo()
    memoria = MemoriaConversas(banco)
    sessao = memoria.iniciar(ana)
    memoria.registrar(ana, sessao, 'Estou falando do cliente Alfa.', '[FIXTURE LOCAL] Entendi.')
    for numero in range(6):
        memoria.registrar(ana, sessao, f'Assunto {numero}', '[FIXTURE LOCAL] Resposta.')
    janela = memoria.mensagens(ana, sessao)
    assert len(janela) == 12 and 'Alfa' not in str(janela)
    recuperados = memoria.buscar_resumos(ana, sessao, 'Qual cliente mencionei?')
    assert recuperados[0]['resumo_extrativo'] == 'Estou falando do cliente Alfa.'
    print('Janela:', len(janela), 'mensagens; recortes:', recuperados)


## 2. Reabrir e separar conversas
A busca verifica empresa, usuário e ID da sessão antes da pontuação lexical. `/nova` cria outra sessão sem reutilizar os recortes anteriores.


In [ ]:
with Banco(caminho) as banco:
    memoria = MemoriaConversas(banco)
    assert memoria.buscar_resumos(ana, sessao, 'Qual cliente?')
    try:
        memoria.buscar_resumos(bruno, sessao, 'Qual cliente?')
    except PermissionError:
        print('Outro usuário bloqueado, como esperado.')
    else:
        raise AssertionError('Falha de isolamento')
    memoria.encerrar(ana, sessao)
    nova = memoria.iniciar(ana)
    assert memoria.buscar_resumos(ana, nova, 'Qual cliente?') == []
    print('Nova conversa sem recortes:', nova)
pasta.cleanup()


## 3. Falha real e correção local
Renan escreveu `cleinte alfa` e depois perguntou por `cliente`; a busca exata retornou zero recortes. O teste abaixo reproduz o erro de digitação e mostra a correspondência aproximada. O notebook não substitui o reteste da resposta real do agente.


In [ ]:
from secretaria.roteamento import roteador_pedido
with tempfile.TemporaryDirectory() as pasta_teste:
    with Banco(Path(pasta_teste) / 'tipos.sqlite3') as banco:
        banco.preparar_demo()
        memoria = MemoriaConversas(banco)
        sessao_teste = memoria.iniciar(ana)
        memoria.registrar(ana, sessao_teste, 'Estou falando do cleinte alfa', '[FIXTURE LOCAL] Entendi.')
        for numero in range(6):
            memoria.registrar(ana, sessao_teste, f'Oi {numero}', '[FIXTURE LOCAL] Olá.')
        achados = memoria.buscar_resumos(ana, sessao_teste, 'qual cliente eu mencionei agora pouco?')
        assert achados[0]['tipo_correspondencia'] == 'aproximada'
        assert roteador_pedido.invoke({'pergunta': 'Estou falando do cleinte alfa'})['rota'] == 'cliente'
        print(achados)


## Teste real ainda necessário
Na conversa que apresentou a falha, pergunte novamente `Qual cliente eu mencionei agora pouco?` após reiniciar o programa. Confira a linha `Recortes antigos recuperados dos turnos:` e a resposta. A busca por palavras semelhantes ainda pode falhar com paráfrases; FAISS fica para a fase 4.
